# 🧹 02 — Data Preprocessing & Feature Engineering

**Fraud Detection Capstone — Sprint 1: Data Understanding & ML Baseline**

## 🎯 Goal
Turn the raw dataset into leak-free, model-ready train/validation/test splits, with scaling and a couple of engineered features.


> ⚠️ **Data source note:** same as `01_eda.ipynb` — this notebook uses the real dataset at `data/creditcard.csv` if present, otherwise a synthetic placeholder with the same schema. The printed `data_source` below states which one was actually used. **Use the real dataset for final Sprint 1 evidence.**


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from data_utils import load_fraud_data

SEED = 42
np.random.seed(SEED)

df, data_source = load_fraud_data()
print(f"Data source: {data_source}")
print(f"Shape: {df.shape}")


Data source: REAL dataset (data/creditcard.csv, 5,009 rows)
Shape: (5009, 31)


## 0. Remove Duplicate Rows

The EDA notebook flagged duplicate rows. Exact duplicates are dropped here, before any split, so the same transaction can't leak across train/val/test.


In [2]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_after = len(df)
print(f"Dropped {n_before - n_after} duplicate rows ({n_before:,} -> {n_after:,})")
print(df['Class'].value_counts())


Dropped 0 duplicate rows (5,009 -> 5,009)
Class
0    5000
1       9
Name: count, dtype: int64


## 1. Train / Validation / Test Split (Leak-Free)

The split happens **before** any scaling or feature engineering is fit, and it is **stratified** on `Class` so the (already tiny) fraud class is represented proportionally in every split. Nothing about the val/test sets is used to fit any transformation — that's decided using the training set only, then *applied* to val/test.


In [3]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)

for name, y_split in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    fraud_rate = y_split.mean() * 100
    print(f"{name}: {len(y_split):>7,} rows | fraud rate: {fraud_rate:.4f}%")


Train:   3,506 rows | fraud rate: 0.1711%
Validation:     751 rows | fraud rate: 0.1332%
Test:     752 rows | fraud rate: 0.2660%


## 2. Feature Engineering (on Raw Values)

Two lightweight engineered features, computed on the **raw, unscaled** `Time`/`Amount` — this matters for `Amount_log`, since `log1p` needs a real dollar amount, not a standardized value (which can be negative and would break the log transform):

- **`Hour`** — `Time` is seconds elapsed since the first transaction; converting to hour-of-day (mod 24) may capture time-of-day fraud patterns more directly than raw seconds.
- **`Amount_log`** — log1p transform of the raw `Amount`, since transaction amounts are heavily right-skewed (see EDA notebook).


In [4]:
def add_engineered_features(X):
    X = X.copy()
    X["Hour"] = (X["Time"] // 3600) % 24
    X["Amount_log"] = np.log1p(X["Amount"])  # raw Amount, always >= 0
    return X

X_train_fe = add_engineered_features(X_train)
X_val_fe = add_engineered_features(X_val)
X_test_fe = add_engineered_features(X_test)

print(f"Feature count: {X_train.shape[1]} -> {X_train_fe.shape[1]}")
print(f"NaNs in Amount_log: {X_train_fe['Amount_log'].isnull().sum()}")
X_train_fe[["Hour", "Amount_log"]].describe()


Feature count: 30 -> 32
NaNs in Amount_log: 0


,Hour,Amount_log
count,3506.000000,3506.000000
mean,11.606959,4.161951
std,6.925287,0.936853
min,0.000000,0.378436
25%,6.000000,3.628731
50%,12.000000,4.286135
75%,18.000000,4.842138
max,23.000000,6.471852


## 3. Scaling

`V1`–`V28` are already PCA-transformed (roughly standardized by construction), but `Time` and `Amount` are raw and on very different scales, so they're standardized here — **after** feature engineering, so `Amount_log` above used the real dollar amount. The scaler is **fit on the training set only**, then applied (transform, not fit) to validation and test.


In [5]:
scale_cols = ["Time", "Amount"]

scaler = StandardScaler()
X_train_fe[scale_cols] = scaler.fit_transform(X_train_fe[scale_cols])
X_val_fe[scale_cols] = scaler.transform(X_val_fe[scale_cols])
X_test_fe[scale_cols] = scaler.transform(X_test_fe[scale_cols])

X_train_fe[scale_cols].describe()


,Time,Amount
count,3.506000e+03,3.506000e+03
mean,2.938639e-16,4.357293e-17
std,1.000143e+00,1.000143e+00
min,-1.705282e+00,-1.208888e+00
25%,-8.706095e-01,-7.236758e-01
50%,-3.864545e-02,-2.543447e-01
75%,8.889489e-01,4.700898e-01
max,1.755906e+00,7.438223e+00


## 4. Save Processed Splits

In [6]:
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

X_train_fe.assign(Class=y_train.values).to_csv(out_dir / "train.csv", index=False)
X_val_fe.assign(Class=y_val.values).to_csv(out_dir / "val.csv", index=False)
X_test_fe.assign(Class=y_test.values).to_csv(out_dir / "test.csv", index=False)

print(f"Saved processed splits to: {out_dir.resolve()}")
for f in sorted(out_dir.glob("*.csv")):
    print(f" - {f.name}: {pd.read_csv(f).shape}")


Saved processed splits to: /home/claude/sprint1/data/processed
 - test.csv: (752, 33)
 - train.csv: (3506, 33)
 - val.csv: (751, 33)


## 📝 Preprocessing Summary

- Split first (stratified, 70/15/15), scaled and engineered features fit on training data only — no leakage into validation or test.
- `Time` and `Amount` standardized; `Hour` and `Amount_log` added as engineered features.
- Processed splits saved to `data/processed/` for the next notebook.

**Next:** `03_baseline_ml_models.ipynb` — baseline classifier and a first neural network.
